In [ ]:
# Kerakli kutubxonalarni yuklash
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
# Datasetni o'qish
data = pd.read_csv('/content/train (1).csv')

In [ ]:
# Ma'lumotlar o'lchami va umumiy ma'lumotlar (EDA)
print(data.shape)
data.info()

In [ ]:
data.head()

In [ ]:
# Asosiy statistik ko'rsatkichlar
data.describe()

In [ ]:
# Omon qolganlar va sinflar bo'yicha taqsimot (indekslar to'g'rilandi: 0 - halok bo'lgan, 1 - omon qolgan)
all_survived = data.Survived.value_counts(normalize=True)
print(f"Omon qolmaganlar: {all_survived[0]*100:.2f} %")
print(f"Tiriklar: {all_survived[1]*100:.2f} %")
print(f"Omonlar soni: {data.Survived.value_counts()[1]}")
print(data.Pclass.value_counts())

In [ ]:
# Jins va Pclass bo'yicha omon qolish ehtimolligi
print(data.groupby('Sex')["Survived"].mean())
print(data.groupby('Pclass')["Survived"].mean())

In [ ]:
# 12 yoshgacha bolalarning omon qolish ulushi
data.groupby(data['Age'] < 12)['Survived'].mean()

In [ ]:
# Hamroh bilan kelgan yo'lovchilarning omon qolish ulushi (mantiqiy qavslar to'g'rilandi)
data.groupby((data['SibSp'] > 0) | (data['Parch'] > 0))['Survived'].mean()

In [ ]:
# Jins va Pclass kesishmasida omon qolish matritsasi
pd.crosstab(data['Sex'], data['Pclass'], values=data['Survived'], aggfunc='mean')

In [ ]:
# Korrelyatsiyalar tahlili
print(data[['Pclass', 'Fare']].corr())
print(data[['Pclass', 'Survived']].corr())

In [ ]:
# Yetishmayotgan qiymatlar foizi (len(df) o'rniga to'g'ri data hajmi olindi)
(data.isnull().sum() / len(data) * 100).round(1)

In [ ]:
# Bo'sh qiymatlarni tozalash va to'ldirish
data = data.drop('Cabin', axis=1)                                # 77% bo'sh bo'lgan ustunni o'chirish
data['Age'] = data['Age'].fillna(data['Age'].median())           # Yoshni median bilan to'ldirish
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])  # Embarked dagi 2 ta bo'sh qiymatni moda bilan to'ldirish

# Tozalashdan keyingi holat
(data.isnull().sum() / len(data) * 100).round(1)

In [ ]:
# Fare ustunidagi keskin og'ishlar (Outliers) ni aniqlash
Q1, Q3 = data["Fare"].quantile([0.25, 0.75])
IQR = Q3 - Q1
out = data[data["Fare"] > (Q3 + 1.5 * IQR)]
print(f"Outlierlar soni: {len(out)}")

In [ ]:
# Feature Engineering: Oila hajmi va yolg'iz sayohat qilganlik belgisi (xato to'g'rilandi)
data['family_size'] = data['SibSp'] + data['Parch'] + 1
data['is_alone'] = (data['family_size'] == 1).astype(int)

print(data['family_size'].describe())
print(data['is_alone'].value_counts())

In [ ]:
# Feature Engineering: Ismlardan unvonlarni ajratish (Mr, Mrs, Miss, Master...)
data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
data['Title'] = data['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Fare anomaliyalarini yumshatish uchun log-transformatsiya
data['Fare_Log'] = np.log1p(data['Fare'])

In [ ]:
# Encoding (Kategorik belgilarni raqamga o'tkazish)
le = LabelEncoder()
data['Sex_encoded'] = le.fit_transform(data['Sex'])

# One-Hot Encoding (drop_first=True bilan multicollinearity oldi olinadi)
data = pd.get_dummies(data, columns=['Embarked', 'Title'], drop_first=True, dtype=int)

In [ ]:
# Scaling: Raqamli ustunlarni standartlashtirish
scaler = StandardScaler()
data[['Age_scaled', 'Fare_scaled']] = scaler.fit_transform(data[['Age', 'Fare_Log']])

# Modelga kiritilmaydigan ortiqcha ustunlarni olib tashlash
drop_cols = ['PassengerId', 'Name', 'Sex', 'Ticket', 'Age', 'Fare', 'Fare_Log']
data_clean = data.drop(columns=drop_cols)

print(data_clean.info())
data_clean.head()